# AI-Powered Bag Surveillance System 



## Step 0 — Create project folder structure

In [ ]:
import os

PROJECT_DIR = "/content/bag_surveillance"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

folders = [
    "config", "detection", "tracking", "association",
    "event_detection", "reid", "database", "api", "utils",
    "tests", "models", "sample_data", "outputs",
]
for f in folders:
    os.makedirs(f, exist_ok=True)

print("Project root:", os.getcwd())
print("Created folders:", folders)


Project root: /content/bag_surveillance
Created folders: ['config', 'detection', 'tracking', 'association', 'event_detection', 'reid', 'database', 'api', 'utils', 'tests', 'models', 'sample_data', 'outputs']


## Step 1 — Install dependencies

Colab already ships `torch` + a CUDA runtime, so this mainly adds `ultralytics`
(YOLOv8) on top. GPU runtime is recommended: **Runtime → Change runtime type → GPU**.


In [ ]:
!pip install -q ultralytics opencv-python-headless

import torch
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


Torch: 2.11.0+cu128 | CUDA available: True


## Step 2 — Scaffold the package (`__init__.py` files)

These mark every folder as an importable Python package. The later-phase
folders (`tracking/`, `association/`, etc.) are created empty now and will be
filled in by the next notebooks in this series.


In [ ]:
init_dirs = [
    "config", "detection", "tracking", "association",
    "event_detection", "reid", "database", "api", "utils", "tests",
]
for d in init_dirs:
    path = os.path.join(d, "__init__.py")
    if not os.path.exists(path):
        open(path, "w").close()

print("Package __init__.py files created for:", init_dirs)


Package __init__.py files created for: ['config', 'detection', 'tracking', 'association', 'event_detection', 'reid', 'database', 'api', 'utils', 'tests']


## Step 3 — Configuration module (`config/settings.py`)

Central place for every threshold used across all phases (detection confidence,
tracking buffers, ownership distance, abandonment timeout, re-id similarity, DB
url, API host/port). Phase 1 only *uses* the `DetectionConfig` section, but the
rest is defined now so later phases import the same `settings` object without
changes here.


In [ ]:
%%writefile config/settings.py
"""
Central configuration for the Bag Surveillance System.

All tunable thresholds, paths, and model parameters live here so that
later phases (tracking, association, event detection, re-id) can import
one consistent settings object instead of scattering magic numbers
through the codebase.
"""

from dataclasses import dataclass, field
from pathlib import Path
from typing import List


# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
BASE_DIR = Path(__file__).resolve().parent.parent
MODELS_DIR = BASE_DIR / "models"
OUTPUTS_DIR = BASE_DIR / "outputs"
SAMPLE_DATA_DIR = BASE_DIR / "sample_data"
DB_DIR = BASE_DIR / "database"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DATA_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Detection (Phase 1)
# ---------------------------------------------------------------------------
@dataclass
class DetectionConfig:
    model_weights: str = "yolov8s.pt"
    confidence_threshold: float = 0.45
    iou_threshold: float = 0.45
    device: str = "auto"

    # 0: person | 1: bicycle, 2: car, 3: motorcycle | 24: backpack, 26: handbag, 28: suitcase
    person_class_ids: List[int] = field(default_factory=lambda: [0])
    vehicle_class_ids: List[int] = field(default_factory=lambda: [1, 2, 3])
    bag_class_ids: List[int] = field(default_factory=lambda: [24, 26])

    label_map: dict = field(default_factory=lambda: {
        0: "person",
        1: "bicycle",
        2: "car",
        3: "motorcycle",
        24: "backpack",
        26: "handbag",

    })
    inference_size: int = 640


# ---------------------------------------------------------------------------
# Tracking (Phase 2)
# ---------------------------------------------------------------------------
@dataclass
class TrackingConfig:
    track_thresh: float = 0.5       # detection confidence to start a track
    track_buffer: int = 30          # frames to keep a "lost" track alive
    match_thresh: float = 0.8       # IoU threshold for matching
    frame_rate: int = 30


# ---------------------------------------------------------------------------
# Association (Phase 3)
# ---------------------------------------------------------------------------
@dataclass
class AssociationConfig:
    max_owner_distance_px: float = 150.0   # max center distance to count as "near"
    ownership_confirm_frames: int = 20      # consecutive proximate frames to confirm ownership
    ownership_lost_seconds: float = 3.0     # owner missing this long => bag becomes "unattended"
    trajectory_window: int = 15             # frames used for trajectory similarity


# ---------------------------------------------------------------------------
# Event detection (Phase 4)
# ---------------------------------------------------------------------------
@dataclass
class EventConfig:
    abandonment_timeout_seconds: float = 30.0
    bag_static_velocity_px: float = 5.0
    theft_pickup_distance_px: float = 100.0


# ---------------------------------------------------------------------------
# Re-identification (Phase 5)
# ---------------------------------------------------------------------------
@dataclass
class ReidConfig:
    embedding_dim: int = 512
    similarity_threshold: float = 0.65
    gallery_max_size: int = 200


# ---------------------------------------------------------------------------
# Database (SQLite for now; swap DB_URL for Postgres later)
# ---------------------------------------------------------------------------
@dataclass
class DatabaseConfig:
    db_url: str = f"sqlite:///{DB_DIR}/surveillance.db"
    echo: bool = False


# ---------------------------------------------------------------------------
# API
# ---------------------------------------------------------------------------
@dataclass
class APIConfig:
    host: str = "0.0.0.0"
    port: int = 8000


@dataclass
class Settings:
    detection: DetectionConfig = field(default_factory=DetectionConfig)
    tracking: TrackingConfig = field(default_factory=TrackingConfig)
    association: AssociationConfig = field(default_factory=AssociationConfig)
    event: EventConfig = field(default_factory=EventConfig)
    reid: ReidConfig = field(default_factory=ReidConfig)
    database: DatabaseConfig = field(default_factory=DatabaseConfig)
    api: APIConfig = field(default_factory=APIConfig)


settings = Settings()


Overwriting config/settings.py


In [ ]:
from config.settings import settings
print(settings.detection)


DetectionConfig(model_weights='yolov8s.pt', confidence_threshold=0.45, iou_threshold=0.45, device='auto', person_class_ids=[0], vehicle_class_ids=[1, 2, 3], bag_class_ids=[24, 26], label_map={0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 24: 'backpack', 26: 'handbag'}, inference_size=640)


## Step 4 — Detection module

* `detection/schemas.py` — a plain `Detection` dataclass (bbox, confidence, class,
  label) shared by every later module, so tracking/association/event_detection
  never need to know about YOLO internals.
* `detection/detector.py` — `PersonBagDetector`, a thin wrapper around
  `ultralytics.YOLO` restricted to COCO classes `person (0)`, `backpack (24)`,
  `handbag (26)`, `suitcase (28)`.


In [ ]:
%%writefile detection/schemas.py
"""
Data structures shared by the detection module.

Keeping this separate from detector.py means tracking/, association/, and
event_detection/ can import the schema without pulling in the YOLO/ultralytics
dependency.
"""

from dataclasses import dataclass
from typing import Tuple


@dataclass
class Detection:
    """A single bounding-box detection in one frame."""

    # (x1, y1, x2, y2) in pixel coordinates, top-left origin.
    bbox: Tuple[float, float, float, float]

    confidence: float
    class_id: int
    label: str  # human-readable, e.g. "person", "backpack"

    @property
    def center(self) -> Tuple[float, float]:
        x1, y1, x2, y2 = self.bbox
        return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

    @property
    def width(self) -> float:
        return self.bbox[2] - self.bbox[0]

    @property
    def height(self) -> float:
        return self.bbox[3] - self.bbox[1]

    @property
    def area(self) -> float:
        return max(0.0, self.width) * max(0.0, self.height)

    def is_person(self) -> bool:
        return self.label == "person"

    def is_bag(self) -> bool:
        return self.label in ("backpack", "handbag")

    def is_vehicle(self) -> bool:
        return self.label in ("bicycle", "car", "motorcycle")

    def to_dict(self) -> dict:
        return {
            "bbox": list(self.bbox),
            "confidence": round(float(self.confidence), 4),
            "class_id": int(self.class_id),
            "label": self.label,
        }


Overwriting detection/schemas.py


In [ ]:
%%writefile detection/detector.py
"""
YOLOv8-based detector for people and bags.

Phase 1 scope: given a frame (numpy BGR image), return a list of
Detection objects restricted to the classes we care about (person,
backpack, handbag, suitcase).

This wrapper deliberately hides all ultralytics-specific API details
behind a small, stable interface (`PersonBagDetector.detect`) so that
later phases (tracking, association) never need to know we're using
YOLOv8 specifically -- swapping in YOLOv11 later means changing only
this file.
"""

from __future__ import annotations

import logging
from typing import List, Optional

import numpy as np

from config.settings import DetectionConfig, settings
from detection.schemas import Detection

logger = logging.getLogger(__name__)


class PersonBagDetector:
    """Thin wrapper around an Ultralytics YOLO model."""

    def __init__(self, config: Optional[DetectionConfig] = None):
        self.config = config or settings.detection
        self._model = None  # lazy-loaded so importing this module is cheap
        self._device = self._resolve_device(self.config.device)
        self._allowed_class_ids = set(
            self.config.person_class_ids +
            self.config.bag_class_ids +
            self.config.vehicle_class_ids
        )

    # ------------------------------------------------------------------
    # Setup
    # ------------------------------------------------------------------
    @staticmethod
    def _resolve_device(device: str) -> str:
        if device != "auto":
            return device
        try:
            import torch

            return "cuda:0" if torch.cuda.is_available() else "cpu"
        except ImportError:
            return "cpu"

    def _load_model(self):
        if self._model is not None:
            return self._model

        from ultralytics import YOLO

        logger.info(
            "Loading YOLO model '%s' on device '%s'",
            self.config.model_weights,
            self._device,
        )
        model = YOLO(self.config.model_weights)
        self._model = model
        return self._model

    # ------------------------------------------------------------------
    # Inference
    # ------------------------------------------------------------------
    def detect(self, frame: np.ndarray) -> List[Detection]:
        """
        Run detection on a single BGR frame.

        Args:
            frame: HxWx3 numpy array (as read by cv2.VideoCapture).

        Returns:
            List of Detection objects for persons and bags only,
            already filtered by confidence and class.
        """
        if frame is None or frame.size == 0:
            return []

        model = self._load_model()

        results = model.predict(
            source=frame,
            conf=self.config.confidence_threshold,
            iou=self.config.iou_threshold,
            imgsz=self.config.inference_size,
            classes=list(self._allowed_class_ids),
            device=self._device,
            verbose=False,
        )

        detections: List[Detection] = []
        if not results:
            return detections

        result = results[0]
        if result.boxes is None or len(result.boxes) == 0:
            return detections

        boxes_xyxy = result.boxes.xyxy.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()
        class_ids = result.boxes.cls.cpu().numpy().astype(int)

        for bbox, conf, cls_id in zip(boxes_xyxy, confidences, class_ids):
            label = self.config.label_map.get(int(cls_id), str(cls_id))
            detections.append(
                Detection(
                    bbox=tuple(float(v) for v in bbox),
                    confidence=float(conf),
                    class_id=int(cls_id),
                    label=label,
                )
            )

        return detections

    def detect_batch(self, frames: List[np.ndarray]) -> List[List[Detection]]:
        """Convenience batch wrapper (sequential for now; safe for Phase 1)."""
        return [self.detect(frame) for frame in frames]


Overwriting detection/detector.py


In [ ]:
# re-export for convenience, matching detection/__init__.py in the full repo
with open("detection/__init__.py", "w") as f:
    f.write(
        "from .schemas import Detection\n"
        "from .detector import PersonBagDetector\n\n"
        "__all__ = [\"Detection\", \"PersonBagDetector\"]\n"
    )
print("detection/__init__.py written")


detection/__init__.py written


## Step 5 — Utilities (`utils/`)

* `video_io.py` — `VideoReader` / `VideoWriter` thin wrappers over OpenCV
* `drawing.py` — draws bounding boxes + a small HUD (frame index, counts)
* `logger.py` — consistent console logging across all modules


In [ ]:
%%writefile utils/logger.py
"""Project-wide logging setup."""

import logging
import sys


def get_logger(name: str, level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger  # avoid duplicate handlers on re-import

    logger.setLevel(level)
    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)
    logger.propagate = False
    return logger


Overwriting utils/logger.py


In [ ]:
%%writefile utils/video_io.py
"""
Video reading/writing helpers built on OpenCV.

Used by main.py and the Colab notebook to iterate over frames of an
uploaded video file and to write an annotated output video.
"""

from __future__ import annotations

from pathlib import Path
from typing import Generator, Optional, Tuple

import cv2
import numpy as np


class VideoReader:
    """Iterates over frames of a video file."""

    def __init__(self, path: str):
        self.path = str(path)
        self.cap = cv2.VideoCapture(self.path)
        if not self.cap.isOpened():
            raise FileNotFoundError(f"Could not open video file: {self.path}")

        self.fps = self.cap.get(cv2.CAP_PROP_FPS) or 25.0
        self.width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.frame_count = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

    def frames(self) -> Generator[Tuple[int, np.ndarray], None, None]:
        """Yield (frame_index, frame) tuples until the video ends."""
        idx = 0
        while True:
            ok, frame = self.cap.read()
            if not ok:
                break
            yield idx, frame
            idx += 1

    def release(self):
        self.cap.release()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.release()


class VideoWriter:
    """Writes annotated frames to an output video file (mp4)."""

    def __init__(self, path: str, fps: float, frame_size: Tuple[int, int]):
        self.path = str(path)
        Path(self.path).parent.mkdir(parents=True, exist_ok=True)
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        self.writer = cv2.VideoWriter(self.path, fourcc, fps, frame_size)

    def write(self, frame: np.ndarray):
        self.writer.write(frame)

    def release(self):
        self.writer.release()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.release()


def get_video_info(path: str) -> dict:
    """Quick metadata probe without iterating frames."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video file: {path}")
    info = {
        "fps": cap.get(cv2.CAP_PROP_FPS),
        "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "frame_count": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()
    return info


Overwriting utils/video_io.py


In [ ]:
%%writefile utils/drawing.py
from __future__ import annotations
from typing import Iterable, Tuple
import cv2
import numpy as np

from detection.schemas import Detection
from tracking.schemas import TrackedObject

COLOR_PERSON = (255, 144, 30)
COLOR_BAG = (0, 200, 0)
COLOR_TEXT_BG = (0, 0, 0)

def _label_color(label: str) -> Tuple[int, int, int]:
    return COLOR_PERSON if label == "person" else COLOR_BAG

def draw_detections(frame: np.ndarray, detections: Iterable[Detection]) -> np.ndarray:
    annotated = frame.copy()
    for det in detections:
        x1, y1, x2, y2 = (int(v) for v in det.bbox)
        color = _label_color(det.label)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        text = f"{det.label} {det.confidence:.2f}"
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(annotated, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
        cv2.putText(annotated, text, (x1 + 2, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return annotated

def draw_tracked_objects(frame: np.ndarray, tracked_objects: Iterable[TrackedObject]) -> np.ndarray:
    annotated = frame.copy()
    for obj in tracked_objects:
        x1, y1, x2, y2 = (int(v) for v in obj.bbox)
        color = _label_color(obj.label)
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)

        # Now drawing the Track ID alongside the label
        text = f"ID:{obj.track_id} {obj.label}"
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(annotated, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
        cv2.putText(annotated, text, (x1 + 2, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return annotated

def draw_frame_info(frame: np.ndarray, frame_idx: int, num_people: int, num_bags: int) -> np.ndarray:
    text = f"frame {frame_idx} | people: {num_people} | bags: {num_bags}"
    cv2.rectangle(frame, (0, 0), (380, 30), COLOR_TEXT_BG, -1)
    cv2.putText(frame, text, (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
    return frame

Overwriting utils/drawing.py


In [ ]:
with open("utils/__init__.py", "w") as f:
    f.write(
        "from .logger import get_logger\n"
        "from .video_io import VideoReader, VideoWriter, get_video_info\n"
        "from .drawing import draw_detections, draw_frame_info\n\n"
        "__all__ = [\n"
        "    \"get_logger\", \"VideoReader\", \"VideoWriter\", \"get_video_info\",\n"
        "    \"draw_detections\", \"draw_frame_info\",\n"
        "]\n"
    )
print("utils/__init__.py written")


utils/__init__.py written


In [ ]:
%%writefile tracking/schemas.py
"""
Placeholder tracking schema to satisfy drawing.py imports in Phase 1.
"""
from dataclasses import dataclass
from typing import Tuple

@dataclass
class TrackedObject:
    track_id: int
    bbox: Tuple[float, float, float, float]
    label: str

Overwriting tracking/schemas.py


## Step 7 — Upload a sample video

Upload a short CCTV/IP-camera-style clip (a person walking with a bag works
well for testing). If you don't have one, the next cell can generate a
synthetic placeholder clip instead so you can still validate the pipeline.


In [ ]:
from google.colab import files

print("Choose a video file to upload (mp4/avi/mov)...")
uploaded = files.upload()

video_filename = list(uploaded.keys())[0]
video_path = os.path.join("sample_data", video_filename)
os.rename(video_filename, video_path)
print("Saved to:", video_path)


Choose a video file to upload (mp4/avi/mov)...


Saving person-bicycle-car-detection.mp4 to person-bicycle-car-detection.mp4
Saved to: sample_data/person-bicycle-car-detection.mp4


## Step 6 — Pipeline entry point (`main.py`)

`run_phase1()` reads a video frame-by-frame, runs the detector, draws results,
and writes an annotated output video — exactly the same function `main.py`'s
CLI calls, so this notebook and the production script stay in sync.


In [ ]:
"""
Bag Surveillance System — main entry point (With Spatial Overlap, Cleanup, & RAG Output)
"""

from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

from config.settings import settings
from detection.detector import PersonBagDetector
from utils.drawing import draw_detections, draw_frame_info
from utils.logger import get_logger
from utils.video_io import VideoReader, VideoWriter

logger = get_logger("main")


def calculate_iou(bbox1: tuple, bbox2: tuple) -> float:
    """Calculate the Intersection over Union (IoU) of two bounding boxes."""
    x1_1, y1_1, x2_1, y2_1 = bbox1
    x1_2, y1_2, x2_2, y2_2 = bbox2

    x_left = max(x1_1, x1_2)
    y_top = max(y1_1, y1_2)
    x_right = min(x2_1, x2_2)
    y_bottom = min(y2_1, y2_2)

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    bbox1_area = (x2_1 - x1_1) * (y2_1 - y1_1)
    bbox2_area = (x2_2 - x1_2) * (y2_2 - y1_2)
    union_area = float(bbox1_area + bbox2_area - intersection_area)

    if union_area == 0:
        return 0.0

    return intersection_area / union_area


def run_phase1(video_path: str, output_path: str, max_frames: int | None = None) -> dict:
    detector = PersonBagDetector(config=settings.detection)

    total_person_dets = 0
    total_bag_dets = 0
    processed_frames = 0

    last_seen = {}
    cooldown = 2.0
    timeline_events = []

    t_start = time.time()

    with VideoReader(video_path) as reader:
        logger.info(
            "Opened video: %s (%dx%d @ %.1f fps, %d frames)",
            video_path, reader.width, reader.height, reader.fps, reader.frame_count,
        )

        with VideoWriter(output_path, reader.fps, (reader.width, reader.height)) as writer:
            for frame_idx, frame in reader.frames():
                if max_frames is not None and frame_idx >= max_frames:
                    break

                detections = detector.detect(frame)
                current_timestamp = round(frame_idx / reader.fps, 1)

                people = [d for d in detections if d.is_person()]
                objects = [d for d in detections if not d.is_person()]

                matched_people = set()
                matched_objects = set()
                current_frame_labels = set()

                # --- SPATIAL OVERLAP LOGIC (IoU) ---
                iou_threshold = 0.15

                for p in people:
                    for obj in objects:
                        iou = calculate_iou(p.bbox, obj.bbox)
                        if iou > iou_threshold:
                            compound_label = f"person with a {obj.label}"
                            current_frame_labels.add(compound_label)
                            matched_people.add(id(p))
                            matched_objects.add(id(obj))

                for p in people:
                    if id(p) not in matched_people:
                        current_frame_labels.add(p.label)

                for obj in objects:
                    if id(obj) not in matched_objects:
                        current_frame_labels.add(obj.label)

                # --- EVALUATE EVENTS FOR TIMELINE ---
                for label in current_frame_labels:
                    if label not in last_seen or (current_timestamp - last_seen[label]) >= cooldown:
                        timeline_events.append((current_timestamp, label))
                    last_seen[label] = current_timestamp

                total_person_dets += len(people)
                total_bag_dets += len([obj for obj in objects if obj.is_bag()])

                annotated = draw_detections(frame, detections)
                annotated = draw_frame_info(annotated, frame_idx, len(people), total_bag_dets)
                writer.write(annotated)

                processed_frames += 1
                if processed_frames % 50 == 0:
                    logger.info("Processed %d frames...", processed_frames)

    elapsed = time.time() - t_start
    avg_fps = processed_frames / elapsed if elapsed > 0 else 0.0

    # --- CLEANUP TIMELINE EVENTS ---
    clean_events = []
    cleanup_window = 1.0

    for i, current_event in enumerate(timeline_events):
        t, label = current_event

        if "with a" not in label:
            is_redundant = False
            for j, nearby_event in enumerate(timeline_events):
                if i == j:
                    continue
                nearby_t, nearby_label = nearby_event

                if abs(t - nearby_t) <= cleanup_window:
                    if "with a" in nearby_label and label in nearby_label.split(" with a "):
                        is_redundant = True
                        break

            if is_redundant:
                continue

        clean_events.append(current_event)

    # --- PRINT THE SPATIAL TIMELINE ---
    print("\n" + "="*60)
    print("🎬 CHRONOLOGICAL EVENT TIMELINE (Cleaned)")
    print("="*60)

    if not clean_events:
        print("No targeted objects were detected in this video.")
    else:
        grouped_events = {}
        for t, label in clean_events:
            if t not in grouped_events:
                grouped_events[t] = []
            grouped_events[t].append(label)

        for t in sorted(grouped_events.keys()):
            labels = [l.capitalize() for l in grouped_events[t]]

            if len(labels) == 1:
                final_str = f"A {labels[0]}"
            elif len(labels) == 2:
                final_str = f"A {labels[0]} and a {labels[1]}"
            else:
                final_str = "A " + ", a ".join(labels[:-1]) + f", and a {labels[-1]}"

            print(f"⏱️ At {t} seconds: {final_str} appeared")

    print("="*60 + "\n")

    # --- NEW: GENERATE RAG DOCUMENTS ---
    rag_documents = []
    video_filename = Path(video_path).name

    for t, label in clean_events:
        rag_documents.append({
            "page_content": f"A {label} appeared.",
            "metadata": {
                "timestamp_seconds": t,
                "video_source": video_filename
            }
        })

    # Return the summary AND the new RAG documents
    summary = {
        "processed_frames": processed_frames,
        "elapsed_seconds": round(elapsed, 2),
        "avg_fps": round(avg_fps, 2),
        "output_path": str(output_path),
        "rag_documents": rag_documents  # <-- Added here!
    }

    return summary


def parse_args():
    parser = argparse.ArgumentParser(description="Bag Surveillance System — Phase 1")
    parser.add_argument("--video", type=str, required=True)
    parser.add_argument("--output", type=str, default="outputs/annotated.mp4")
    parser.add_argument("--max-frames", type=int, default=None)

    if 'ipykernel' in sys.modules:
        return parser.parse_args(args=["--video", "sample_data/person-bicycle-car-detection.mp4"])
    else:
        return parser.parse_args()


if __name__ == "__main__":
    args = parse_args()
    Path(args.output).parent.mkdir(parents=True, exist_ok=True)

    # Store the output so you can use it in your notebook
    results = run_phase1(args.video, args.output, max_frames=args.max_frames)

    # Optional: Print how many documents were generated
    print(f"✅ Generated {len(results['rag_documents'])} documents for RAG pipeline.")

09:58:28 | INFO     | main | Opened video: sample_data/person-bicycle-car-detection.mp4 (768x432 @ 12.0 fps, 647 frames)
09:58:31 | INFO     | main | Processed 50 frames...
09:58:32 | INFO     | main | Processed 100 frames...
09:58:32 | INFO     | main | Processed 150 frames...
09:58:33 | INFO     | main | Processed 200 frames...
09:58:34 | INFO     | main | Processed 250 frames...
09:58:34 | INFO     | main | Processed 300 frames...
09:58:35 | INFO     | main | Processed 350 frames...
09:58:36 | INFO     | main | Processed 400 frames...
09:58:37 | INFO     | main | Processed 450 frames...
09:58:37 | INFO     | main | Processed 500 frames...
09:58:38 | INFO     | main | Processed 550 frames...
09:58:38 | INFO     | main | Processed 600 frames...

🎬 CHRONOLOGICAL EVENT TIMELINE (Cleaned)
⏱️ At 1.2 seconds: A Person appeared
⏱️ At 16.9 seconds: A Car appeared
⏱️ At 26.6 seconds: A Person with a bicycle appeared
⏱️ At 40.4 seconds: A Person appeared
⏱️ At 44.0 seconds: A Car appeared
⏱️ A

**No video handy?** Run this cell instead to generate a tiny synthetic test clip
(solid background + frame counter — won't trigger real detections, but proves
the pipeline runs end-to-end). Skip this cell if you uploaded a real video above.


In [ ]:
"""import cv2
import numpy as np

synthetic_path = "sample_data/synthetic_test.mp4"
writer = cv2.VideoWriter(synthetic_path, cv2.VideoWriter_fourcc(*"mp4v"), 10.0, (640, 480))
for i in range(60):
    frame = np.full((480, 640, 3), 40, dtype=np.uint8)
    cv2.putText(frame, f"frame {i}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    writer.write(frame)
writer.release()

video_path = synthetic_path
print("Synthetic test video created at:", video_path)


SyntaxError: incomplete input (3040008124.py, line 1)

## Step 9 — Preview the annotated output video

Colab can't always inline-render mp4 reliably depending on codec; this cell
re-encodes with a browser-safe codec via ffmpeg, then displays it inline.


In [ ]:
import os
from IPython.display import HTML
from base64 import b64encode

# 1. Point to the video you just generated
output_path = "outputs/annotated_video.mp4"

# 2. Define where to save the web-friendly version
playable_path = "outputs/annotated_playable.mp4"

# 3. Use FFmpeg to compress and convert the video so the browser can read it
print("Converting video for playback... (this might take a few seconds)")
os.system(f'ffmpeg -y -i "{output_path}" -vcodec libx264 "{playable_path}" -loglevel quiet')

# 4. Read the converted video and display it using HTML
print("Done! Loading video...")
video_bytes = open(playable_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(video_bytes).decode()

HTML(f"""
<video width=640 controls>
      <source src="{data_url}" type="video/mp4">
</video>
""")

Converting video for playback... (this might take a few seconds)
Done! Loading video...


## Step 10 — Download the annotated video (optional)

In [ ]:
from google.colab import files as colab_files
colab_files.download(playable_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from langchain_core.documents import Document

# 1. Grab the raw documents from main.py output
raw_docs = results['rag_documents']

# 2. Fix: Explicitly inject the timestamp into the page text string!
langchain_docs = []
for doc in raw_docs:
    t = doc["metadata"]["timestamp_seconds"]
    content_with_time = f"At {t} seconds: {doc['page_content']}"

    langchain_docs.append(
        Document(page_content=content_with_time, metadata=doc["metadata"])
    )

# Print the first one to verify it looks correct
print("👉 Fixed Document Format:")
print(langchain_docs[5])

👉 Fixed Document Format:
page_content='At 46.2 seconds: A person with a bicycle appeared.' metadata={'timestamp_seconds': 46.2, 'video_source': 'person-bicycle-car-detection.mp4'}


In [ ]:
!pip install -qU langchain langchain-classic langchain-community langchain-huggingface langchain-groq faiss-cpu sentence-transformers

ERROR: Operation cancelled by user


In [ ]:
import os
from getpass import getpass

# Securely enter your Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# --- THE FIX: LangChain v1.0+ moved chains to 'classic' ---
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
# ----------------------------------------------------------

print("⏳ Embedding documents into Vector Database...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(langchain_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

print("⏳ Connecting to Groq LLM...")
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

system_prompt = (
    "You are an AI security assistant. Your job is to analyze video event logs and answer questions. "
    "Use the provided context to answer the user's question accurately. "
    "CRITICAL INSTRUCTION: You MUST always include the exact timestamp (in seconds) in your answer. "
    "If the answer is not in the context, do not make it up. Say 'I do not see that event in the logs.'\n\n"
    "Context:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("✅ Surveillance Chatbot is ready!")

⏳ Embedding documents into Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

⏳ Connecting to Groq LLM...
✅ Surveillance Chatbot is ready!


In [ ]:
# 1. Define your test questions based on the video timeline
test_queries = [
    "When did a bicycle first appear in the video, and was anyone with it?",
    "At what timestamps did cars appear in the frame?",
    "Give me a bulleted chronological summary of all events found in the log.",
    "Were there any airplanes or dogs detected in this footage?"
]

# 2. Run through the queries and print responses
for i, query in enumerate(test_queries, 1):
    print(f"\n🔍 Query {i}: {query}")
    print("-" * 50)

    # Run the query through your RAG chain
    response = rag_chain.invoke({"input": query})

    print(f"🤖 Answer:\n{response['answer']}")
    print("=" * 60)


🔍 Query 1: When did a bicycle first appear in the video, and was anyone with it?
--------------------------------------------------
🤖 Answer:
A bicycle first appeared in the video at 26.6 seconds. A person was with the bicycle.

🔍 Query 2: At what timestamps did cars appear in the frame?
--------------------------------------------------
🤖 Answer:
Cars appeared at the following timestamps: 
- 16.9 seconds
- 44.0 seconds

🔍 Query 3: Give me a bulleted chronological summary of all events found in the log.
--------------------------------------------------
🤖 Answer:
Here's a bulleted chronological summary of the events found in the log:

* 1.2 seconds: A person appeared.
* 16.9 seconds: A car appeared.
* 26.6 seconds: A person with a bicycle appeared.
* 40.4 seconds: A person appeared.
* 44.0 seconds: A car appeared.
* 46.2 seconds: A person with a bicycle appeared.

🔍 Query 4: Were there any airplanes or dogs detected in this footage?
--------------------------------------------------
🤖